
**Concise version:**

To characterize how the human brain's functional network organization — measured through phase-synchronized EEG connectivity — reorganizes across sleep stages, and to determine whether information-theoretic measures of complexity, integration, and directed information flow reveal principled, interpretable signatures of these state transitions.

Slightly expanded version, capturing the "why":

This project asks whether sleep can be understood as a sequence of distinct network states — not just in terms of which regions talk to which (classical connectivity), but in terms of how efficiently and in what direction information moves through the network as the brain passes from wakefulness through light sleep, deep sleep, and REM. Using high-density EEG from healthy adults, the aim is to build brain networks per sleep stage and frequency band, and then ask three linked questions: (1) how does the topology of these networks change (are they more segregated into modules, or more globally integrated, at different stages?), (2) does information flow have a consistent, stage-dependent direction (e.g., does anterior-to-posterior communication dominate in some stages and reverse in others?), and (3) can a graph neural network, trained to distinguish these states, point back — via its attention weights — to the same connections that the classical and information-theoretic measures already flagged as important, as a form of internal validation.

In [ ]:
anphy_sleep/
├── main.ipynb                  # orchestration: calls scripts, inspects outputs, plots
├── config.py                   # paths, band definitions, epoch length, channel list, LOSO folds
│
├── data_io.py                  # Phase 1: EDF loading, montage from .pos, artifact/scoring parsing
├── preprocessing.py            # Phase 1: filtering, epoching, per-subject chunked loop
│
├── connectivity.py             # Phase 2: band-pass + Hilbert + wPLI, graph construction, .npy saving
│
├── graph_metrics.py            # Phase 3.1: clustering, path length, efficiency, modularity, MST
├── entropy_metrics.py          # Phase 3.2: permutation/spectral entropy, graph entropy
├── mutual_information.py       # Phase 3.3: pairwise MI, integration measures
├── transfer_entropy.py         # Phase 3.4: TE computation, net-flow directionality, surrogate testing
├── integration_measures.py     # Phase 3.5: Φ-approximation, community detection, participation coeffs
│
├── stats_analysis.py           # Phase 4: group comparisons, LOSO splitting, correlations
│
├── gnn/
│   ├── dataset.py              # graph → PyG Data objects, per-band and multi-relational variants
│   ├── models.py                # Stage A (per-band GAT), Stage B (R-GCN)
│   ├── train.py                 # training loop, LOSO evaluation
│   └── explain.py               # GNNExplainer, attention-weight extraction
│
├── viz.py                      # scalp-projected connectivity, NetworkX/PyVis topology views
│
└── outputs/
    ├── graphs/                 # per-subject/epoch/band .npy connectivity matrices
    ├── features/               # extracted graph/info-theoretic features (tabular, feeds Phase 4)
    └── figures/

NEW VERSION (24/7/2026):

# Information-Theoretic & Complex-Network Analysis of Sleep EEG (ANPHY-Sleep)

**Goal:** Characterize how brain network organization changes across sleep stages using phase-synchronization graphs and complex-network/graph-theoretic measures — with a GNN layer added for explainability, not classification. Information-theoretic measures that require the raw signal (entropy, mutual information, transfer entropy) are deliberately scoped out of the current pipeline (see "Scope decision" below) but remain part of the project's intended philosophy as a later extension.

**Dataset:** ANPHY-Sleep (OSF, DOI 10.17605/OSF.IO/R26FH, project id `r26fh`) — 29 healthy adults, 83-channel HD-EEG + PSG, overnight recordings, co-registered electrode positions (`.pos`), sleep scoring + artifact matrix provided per subject. **Subject EPCTL08 was removed from the dataset** — skipped everywhere (28 effective subjects).

**Working environment:** Local laptop for Phases 1–4 (CPU/RAM-bound, streaming per-epoch processing); Kaggle (free, ~30 GPU hrs/week) for Phase 5 GNN training once graphs are precomputed.

---

## Scope decision

The original plan treated classical graph theory and information-theoretic measures (entropy, MI, transfer entropy, Φ) as parallel, complementary analyses. In practice, only the graph-theoretic half can be computed from the stored wPLI graphs alone — entropy, MI, and TE all require the raw EEG time series, which the pipeline deliberately discards after each epoch's graphs are built (this is what keeps storage to ~28 small files instead of hundreds of gigabytes of raw signal).

**Decision: stay wPLI/graph-only for now.** Phase 3.2–3.4 (signal-based entropy, MI, transfer entropy) and the Φ-approximation in 3.5 (which needs entropy/MI as inputs) are bookmarked, not implemented — re-enabling them later would require re-reading the raw EDF and either storing the signal (~50GB across the dataset even downsampled) or computing those measures inline during a dedicated re-processing pass. Graph-theoretic analogs that stay within the wPLI-only constraint (graph entropy from the degree distribution, graph-Laplacian spectral entropy — distinct from *signal* spectral entropy) remain in scope and are treated as the information-theoretic side of the current analysis.

---

## Confirmed file formats (from real files, not the readme alone)

- **`.pos`** (electrode positions): first line = channel count (83), then rows of `index / channel_name / x / y / z`, coordinates in cm.
- **Artifact matrix**: per-subject `*_artndxn.mat`, MATLAB v7.3 (HDF5), dataset `artndxn` shaped `(n_epochs, n_channels)`, binary, 1 = clean / 0 = artifact.
- **Sleep scoring `.txt`**: tab/whitespace-delimited, no header, columns `[stage_raw, start_time_sec, duration_sec]`. Stage codes: `W`, `N1`, `N2`, `N3`, `R`, `L` (light on/off marker, not a sleep stage — dropped by default).
- **EDF**: 93 channels, all at 1000 Hz. 83 are EEG (matching the `.pos` file); the other 10 are PSG (`ChEMG1`, `ChEMG2`, `RLEG-`, `RLEG+`, `LLEG-`, `LLEG+`, `ECG1`, `EOG1`, `ECG2`, `EOG2`). A full night at 83 EEG channels in float64 is ~18GB — too large to preload, which drove the streaming design below.
- **Details CSV**: columns Subjects ID, Sex, Age, TRT, TST (min), SE (% TRT), SOL (min), REML (min), WASO (min), N1/N2/N3/R (min).

---

## Data acquisition

Downloaded directly from OSF via the public API (`api.osf.io/v2`), not manual browser downloads — `download_data.py`:
- `download_shared_files()` — one-time download of the artifact matrix zip, `.pos` file, details CSV.
- `download_subject(subject_num, dest_dir)` / `get_available_subjects()` — per-subject download and extraction, skipping EPCTL08 automatically, skipping subjects already extracted.
- `get_subject_edf_path()`, `get_subject_scoring_path()`, `get_subject_artifact_path()` — locate each subject's three needed files after download.

---

## Phase 1 — Data Loading & Preprocessing (streaming, no full-night load)

Per subject, one epoch at a time — never more than one epoch's raw signal in memory:

1. Open the EDF with `pyedflib` (not `mne.io.read_raw_edf`, which attempted to preload the full recording into memory even with `preload=False` and crashed on this dataset's size).
2. For each epoch: read only that time slice via `readSignal(chn, start, n)`.
3. Label with sleep stage from the scoring file; keep only epochs flagged clean in the artifact matrix (`epoch_clean_mask`, strict "all channels clean" mode) and drop `light_on_off` epochs.
4. **No filtering applied at this stage** — an earlier broadband (0.3–45 Hz) + 50 Hz notch filter was removed: both were redundant once redesigned around per-band filtering in Phase 2 (no band reaches 50 Hz; a broadband pre-filter changes nothing once the narrower per-band filter runs anyway).
5. **Downsample 1000 Hz → 200 Hz** (`downsample_signal`, via `scipy.signal.resample_poly`) — 1000 Hz is far more than delta–beta (0.5–30 Hz) needs under the Nyquist theorem; 200 Hz keeps a conservative >3x margin above beta's 30 Hz edge while cutting sample count (and downstream filter/wPLI compute cost) 5x. `resample_poly`'s internal anti-aliasing filter is the only filtering step in the entire pipeline before Phase 2's per-band filters — it's solving a different problem (safe sample-rate reduction) than band isolation, so the two don't overlap or duplicate work.

---

## Phase 2 — Graph Construction (wPLI, per band, per epoch)

For each clean, downsampled epoch, for each of **four** frequency bands — **delta (0.5–4 Hz), theta (4–8 Hz), alpha (8–12 Hz), sigma (12-16 Hz), beta (16–30 Hz)** (sigma dropped from the original five-band list):

1. Band-pass filter (zero-phase Butterworth, `filtfilt`) into that band.
2. Hilbert transform → analytic signal → instantaneous phase per channel.
3. Compute wPLI between every channel pair, row-by-row (not as one full `(83, 83, n_samples)` broadcast, which would need several GB — row-by-row keeps peak memory to a few MB).
4. Result: one 83×83 wPLI matrix per (epoch, band).

**Why wPLI:** weights phase differences by the magnitude of their imaginary component, down-weighting noisy near-zero-lag samples that plain PLI and coherence are vulnerable to.

**Electrode positions and channel identity are node attributes**, not used to define edges — edges are purely functional (wPLI). Kept anatomy and function separate, per the original plan.

---

## Storage: one HDF5 file per subject (revised from per-epoch files)

The original plan called for saving each `(epoch, band)` matrix as a separate `.npy`/`.npz` — at ~900 epochs × 28 subjects, that's ~25,000 files, which is unwieldy on most filesystems (slow to open/list/back up, independent of total byte size). Revised design: **one `output_dir/<subject_id>.h5` file per subject**, containing:
- one dataset per band, shape `(n_clean_epochs, 83, 83)`, gzip-compressed
- `epoch_idx` and `stage` datasets (the metadata that would otherwise need a separate CSV)
- `channel_names` dataset — the fixed row/column order of every band's matrices for that subject, so a matrix index can be mapped back to an electrode name (and, via `.pos`, a 3D position) without re-reading the EDF. Needed both for plotting and as GNN node identity/position features.

`read_subject_graphs(h5_path)` loads a subject's file back into a dict of arrays. `build_global_manifest(output_dir)` scans all subjects' `.h5` files into one lightweight pandas-friendly index (subject/epoch/stage) across the whole dataset, without opening every file to inspect it.

Storage/library note: graphs are kept as **plain NumPy arrays**, not NetworkX or PyTorch Geometric objects — both of those are built transiently, on demand, from the stored arrays (NetworkX for Phase 3 metrics, PyTorch Geometric for Phase 5 GNN training), rather than committing storage to either library's heavier object format.

---

## Phase 3 — Graph‑theoretical analysis of EEG functional networks (wPLI-only scope)

Computed per epoch / band / stage / subject, from the stored wPLI graphs:

**3.1 Classical graph topology metrics (NetworkX)** 

Per subject × epoch × band --> across 28 subjects:

- Classical and dynamic graph analysis: weighted clustering, characteristic path length, global and local efficiency, modularity and community structure, small-worldness (normalized by null graphs), node strength/weighted degree, betweenness centrality, participation coefficient.  Dynamic network organization: temporal trajectories of mean wPLI and selected graph metrics, consecutive-epoch adjacency-matrix similarity, stage-centred and transition-centred trajectories, hub persistence, if feasible. Optional, restricted flexibility analysis.

- **MST as a secondary robustness check** (Kruskal's), not the primary graph representation — confirms stage differences in richer metrics aren't artifacts of raw connectivity strength changing.

**3.2 Graph-theoretic entropy/complexity** 
- Graph entropy (degree distribution).
- Graph-Laplacian spectral entropy (a property of the graph's Laplacian eigenvalue distribution — distinct from, and not to be confused with, *signal* spectral entropy, which is out of scope; both happen to share the name "spectral entropy").
- ~~Permutation entropy / signal spectral entropy per channel~~ — bookmarked, needs raw signal.

**3.3 Integration** — partially in scope
- Global integration (efficiency + modularity balance) — ready now.
- ~~Pairwise mutual information as a complementary connectivity view~~ — bookmarked, needs raw signal.

**3.4 Directed information flow — Information-oriented summaries**

From wPLI alone:

- Graph entropy from node strengths
- Node connectivity entropy
- Integration–segregation profiles based on efficiency, modularity, clustering, and participation

This preserves the original “information-theoretic characterization” aim without claiming directed flow.

**3.5 Advanced / IIT-inspired & additional graph measures**
- **Φ-approximation** — bookmarked (needs entropy/MI as inputs).
- **Community structure** (Louvain/Newman modularity) — ready now.
- **Participation coefficients** (cross-module hub vs. within-module specialist) — ready now.
- **Additional graph-only measures available, not yet in the core plan** — worth considering as extensions: weighted degree/node strength, eigenvector centrality, assortativity, rich-club coefficient, k-core decomposition, algebraic connectivity (Fiedler value), edge betweenness, graph edit distance/similarity between epochs or stages.

**FINAL PROTOCOL:**

1. Save primary excluded-beta static metrics.

2. Threshold stage-mean graphs:
   10%, 20%, 30% + 85%, 90%.

3. Create threshold-stability plots / AUC summaries.

4. Run MST robustness on stage-mean graphs.

5. Run primary static statistics:
   mean wPLI, clustering, modularity.

6. Report the full-beta comparison as beta sensitivity evidence.

7. Run per-epoch mean-wPLI trajectories and
   consecutive-epoch adjacency similarity.

8. Add windowed clustering/modularity only if feasible.

9. Calculate stage-mean graph entropy,
   especially Laplacian spectral entropy.

10. Calculate participation coefficient and nodal strength
    only for globally supported stage effects.

11. Keep TE, MI, signal entropy, and Phi-type methods
    as a later separate raw-signal project.
---

## Phase 4 — Statistics & Interpretation

- Group-level comparisons across stages (Wake/N1/N2/N3/REM), subject as random effect, multiple-comparison correction.
- Correlate graph-theoretic features with sleep quality metrics (efficiency, %N3, arousal index) and demographics.
- Dynamic trajectories across the night.
- Visualization: scalp-projected connectivity (using stored `channel_names` + `.pos` coordinates), NetworkX/PyVis for abstract topology views.
- **Validation strategy: subject-level LOSO** (Leave-One-Subject-Out) as primary evaluation, given N=28 effective subjects vs. thousands of epoch-level graphs.

**Note: Recommended statistical hierarchy:** Stage-level static metrics: use Bayes, a repeated-measures ANOVA or, preferably, a linear mixed-effects model. Outcome: e.g. global efficiency. Fixed effects: stage, band, stage × band. Random effect: subject. Report the F or model coefficient/test, p-value, and effect size. Planned two-stage contrasts: paired t-tests. Example: sigma global efficiency in N2 vs Wake. Report t, degrees of freedom, corrected p-value, and paired Cohen’s dz. Dynamic measures: use a mixed-effects model if you compare stages or transitions. Example: consecutive-epoch similarity predicted by stage, with subject as a random intercept. For transition trajectories, include relative time, transition type, and their interaction if the data support it. Effect sizes: Paired comparisons: Cohen’s dz. ANOVA-style effects: partial η2 or generalized η2. Mixed models: standardized coefficient, marginal/conditional R2, and confidence intervals. Multiple comparisons: apply FDR correction within a pre-defined family—for example, across the five bands for one metric and one planned stage effect. This controls false positives without becoming excessively conservative.

---

## Phase 5 — GNN (secondary, explainability-focused, staged)

**Stage A — per-band GNN (build this first):**
- One GAT per frequency band, input = that band's wPLI graph.
- **Node features: `.pos` 3D coordinates + degree/centrality only** — local entropy dropped from the original plan: it needs raw signal (contradicts the wPLI-only scope decision) and, separately, would partly weaken the explainability validation below by feeding the model a quantity derived from the same topology analysis it's later checked against.
- Task: sleep-stage classification or self-supervised embedding — used as a *validation/explainability tool*, not the project's main output.
- Explainability: GNNExplainer, attention-weight visualization on scalp — cross-checked against Phase 3 findings (e.g., does the model's attention independently land on high-centrality/high-participation nodes Phase 3 also flagged, without having been given that information directly).
- Evaluated with subject-level LOSO.

**Stage B — multi-relational GNN (upgrade once Stage A works):**
- Same 83 nodes, 4 parallel edge types (one per band, per the revised band list) — direct GNN counterpart of the per-band graph structure from Phase 2.
- Architecture: Relational GCN (R-GCN) or multi-relational GAT (`RGCNConv` in PyTorch Geometric).
- Lets the model learn cross-band interactions and per-edge-type attention.

**Compute:** run on Kaggle (free GPU) once graphs are precomputed — no raw EEG needed at this stage.

---

## Phase 6 — Validation, Visualization, Reporting

- Sensitivity analyses: different thresholds, sensor vs. ROI level.
- Honest limitations: N=28 effective subjects (EPCTL08 removed), single night per subject, wPLI/graph-only scope (signal-based information-theoretic measures deferred) — framed as a rigorous pilot-scale, high-density normative study of network reorganization rather than a clinical/generalizable tool.
- Position relative to related work: complements Hypno-PC (temporal/state modeling) and AnySleep (staging performance) rather than competing with them; methodologically related to but distinct from PLI/MST clinical-comparison studies.

---

## Key decisions locked in

| Decision point | Choice |
|---|---|
| Edge definition | wPLI (phase synchronization), not distance-based, not plain coherence |
| Node attributes | `.pos` 3D coordinates + degree/centrality (not local entropy — needs raw signal) |
| Band handling | Separate graph per band, no collapsing — **4 bands**: delta, theta, alpha, beta (sigma dropped) |
| Filtering | None on the raw signal — removed as redundant given per-band filtering; only filtering left is `resample_poly`'s internal anti-aliasing filter during downsampling |
| Downsampling | 1000 Hz → 200 Hz (`resample_poly`, Fraction-based ratio) — Nyquist margin over beta's 30 Hz edge |
| **Scope** | **wPLI/graph-only for now** — entropy, MI, transfer entropy, Φ bookmarked, deferred (need raw signal, not stored) |
| Storage | One HDF5 file per subject (graphs + `epoch_idx` + `stage` + `channel_names` together), not per-epoch files |
| Data acquisition | OSF public API via `requests`, not manual download; subject-by-subject with skip logic |
| Subject exclusion | EPCTL08 removed from the dataset — skipped everywhere (28 effective subjects) |
| MST | Secondary robustness check only, not primary graph representation |
| Cross-validation | Subject-level LOSO, given N=28 |
| GNN role | Explainability/validation tool, not the project's primary output |
| GNN band structure | Per-band GNN first, multi-relational GNN as upgrade |
| Compute | Local laptop for Phases 1–4 (streaming, per-subject), Kaggle free GPU for Phase 5 |